In [ ]:
import json

import pandas as pd

from pfas_trace import DATASET_FILE, DATASET_PATH

In [ ]:
# ~4s expected runtime
df = pd.read_parquet(DATASET_FILE)
df = df.loc[(df["country"] == "France") & (df["category"] == "Measurement")]
df.shape

In [ ]:
df[df["pfas_values"].str.len() > 2].head(3)

In [ ]:
# ~1m 45s expected runtime on first run, ~2s afterwards
DATASET_PFAS_VALUES = DATASET_PATH / "extract.pfas-measurements.parquet"
if not DATASET_PFAS_VALUES.exists():
    df["pfas_values"] = df["pfas_values"].apply(json.loads)
    df_exploded = df.explode("pfas_values").reset_index(drop=True)
    json_df = pd.json_normalize(df_exploded["pfas_values"])
    df_pfas = pd.concat([df_exploded[
                             ["category", "lat", "lon", "name", "city", "country", "type", "sector", "source_type",
                              "data_collection_method", "source_text", "source_url", "dataset_id", "dataset_name",
                              "pfas_sum", "details", "matrix", "date", "year"]], json_df], axis=1)
    df_pfas.to_parquet(DATASET_PFAS_VALUES)
else:
    df_pfas = pd.read_parquet(DATASET_PFAS_VALUES)

df_pfas.shape  # should be >=25'712'660 rows

In [ ]:
print("Amount of different chemicals: ", df_pfas["substance"].nunique())
df_pfas.head(10)

In [ ]:
df_pfas["substance"].value_counts()

In [ ]:
# approximate ~5m locations via coordinate rounding (grid, not a true radius):
# 0.0001 deg ~= 11m latitude / ~7-8m longitude across France's latitude range
df["lat_bin"] = df["lat"].round(4)
df["lon_bin"] = df["lon"].round(4)

location_counts = (
    df.groupby(["year", "lat_bin", "lon_bin"])
    .size()
    .reset_index(name="measurement_count")
    .sort_values("measurement_count", ascending=False)
)
location_counts